In [27]:
import os
import json
import time
import gzip
import shutil
from typing import Dict, List, Optional, Tuple
from urllib.parse import urlparse, unquote
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import requests
class TomTomTrafficDataCollector:
    """
    Thu thập dữ liệu lịch sử từ TomTom Traffic Stats - Area Analysis
    - Địa điểm: Quận 1, TP.HCM (geometry mặc định bên dưới)
    - Khung giờ: 24 slot 15 phút (06:00–09:00 và 13:00–16:00), chỉ MON–FRI
    - Tự động ACCEPT nếu NEED_CONFIRMATION
    - Tải JSON/GeoJSON/SHAPE (ZIP), tự giải nén gzip cho JSON/GeoJSON
    """
    BASE = "https://api.tomtom.com/traffic/trafficstats"
    TIMEZONE = "Asia/Ho_Chi_Minh"
    def __init__(self, api_key: str):
        # Tìm thư mục cha có tên "Urban-Traffic-Links" từ cwd trở lên
        cwd = Path(os.getcwd()).resolve()
        root = None
        if cwd.name.lower() == "urban-traffic-links":
            root = cwd
        else:
            for p in cwd.parents:
                if p.name.lower() == "urban-traffic-links":
                    root = p
                    break
        if root is None:
            # fallback: giả sử bạn mở terminal ở "C:\AI\Specialized_Project2_github"
            # thì ghép cứng với "Urban-Traffic-Links"
            root = cwd / "Urban-Traffic-Links"

        self.output_dir = root / "data" / "raw" / "tomtom_stats"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.api_key = api_key
        print(f"💾 Output directory set to: {self.output_dir}")

    @staticmethod
    def _build_15min_slots(blocks: List[Tuple[str, str]]) -> List[Dict]:
        """
        Tạo danh sách timeSets 15' rời rạc.
        blocks: danh sách (start, end) theo HH:MM, ví dụ [("06:00","09:00"),("13:00","16:00")]
        Trả về: list timeSets (tối đa 24 theo giới hạn Area Analysis)
        """
        def minutes(hhmm: str) -> int:
            h, m = map(int, hhmm.split(":"))
            return h * 60 + m

        def hhmm_from_minutes(total: int) -> str:
            h = total // 60
            m = total % 60
            return f"{h:02d}:{m:02d}"

        time_sets = []
        idx = 0
        for start, end in blocks:
            s = minutes(start)
            e = minutes(end)
            cur = s
            while cur < e:
                slot_start = hhmm_from_minutes(cur)
                slot_end = hhmm_from_minutes(min(cur + 15, e))
                name = f"T{slot_start.replace(':','')}_{slot_end.replace(':','')}"
                time_sets.append({
                    "name": name,
                    "timeGroups": [{
                        "days": ["MON", "TUE", "WED", "THU", "FRI"],
                        "times": [f"{slot_start}-{slot_end}"]
                    }]
                })
                idx += 1
                cur += 15
        # Giới hạn 24 timeSets theo API
        return time_sets[:24]

    # -------------------- CREATE JOB --------------------
    def create_area_analysis_job(
        self,
        geometry: Dict,
        date_from: str,
        date_to: str,
        job_name: str = "HCM D1 15min Slots",
        frcs: List[str] = None,
        probe_source: str = "ALL",
        average_sample_size_threshold: int = 0,
    ) -> Optional[str]:
        """
        geometry: GeoJSON Polygon/MultiPolygon (EPSG:4326)
        date_from/date_to: 'YYYY-MM-DD' (tối đa 1 năm)
        frcs: danh sách FRC ('0'..'7'); mặc định lấy đầy đủ
        probe_source: PASSENGER | TELEMATICS | ALL
        average_sample_size_threshold: nếu >0 và sample thấp hơn ngưỡng, job sẽ cần xác nhận/REJECT
        """

        if frcs is None:
            frcs = ["0", "1", "2", "3", "4", "5", "6", "7"]

        # 24 slot 15' cho 06:00–09:00 và 13:00–16:00
        time_sets = self._build_15min_slots([("06:00", "09:00"), ("13:00", "16:00")])

        body = {
            "jobName": job_name,
            "distanceUnit": "KILOMETERS",
            "network": {
                "name": "District 1 HCMC",
                "geometry": geometry,
                "timeZoneId": self.TIMEZONE,
                "frcs": frcs,
                "probeSource": probe_source
            },
            "dateRange": {
                "name": "Weekdays",
                "from": date_from,
                "to": date_to,
                "excludedDaysOfWeek": ["SAT", "SUN"]
            },
            "timeSets": time_sets,
            "averageSampleSizeThreshold": average_sample_size_threshold
        }

        url = f"{self.BASE}/areaanalysis/1?key={self.api_key}"
        try:
            resp = requests.post(url, headers={"Content-Type": "application/json"}, json=body, timeout=60)
            resp.raise_for_status()
            jr = resp.json()
            if jr.get("responseStatus") == "OK" and jr.get("jobId"):
                job_id = str(jr["jobId"])
                print(f"✅ Job created: {job_id}")
                return job_id
            print("❌ Create job failed:", jr)
            return None
        except requests.HTTPError as e:
            print("❌ HTTPError:", getattr(e.response, "status_code", "?"), getattr(e.response, "text", ""))
            return None
        except Exception as e:
            print("❌ Exception when creating job:", e)
            return None
    
    def _status_url(self, job_id: str) -> str:
        return f"{self.BASE}/status/1/{job_id}?key={self.api_key}"

    def check_job_status(self, job_id: str) -> dict:
        try:
            r = requests.get(self._status_url(job_id), timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print("❌ Status check error:", e)
            return {}

    def _accept_job(self, job_id: str) -> bool:
        url = f"{self.BASE}/status/1/{job_id}/accept?key={self.api_key}"
        try:
            r = requests.post(url, timeout=30)
            r.raise_for_status()
            print("👍 Accepted job for processing.")
            return True
        except Exception as e:
            print("❌ Accept failed:", e, getattr(r, "text", ""))
            return False

    
    def wait_for_job_completion(self, job_id: str, max_wait_minutes: int = 60, poll_seconds: int = 20) -> Optional[dict]:
        """
        Chờ tới khi DONE / ERROR / REJECTED / CANCELLED.
        Nếu NEED_CONFIRMATION -> tự động ACCEPT 1 lần.
        """
        print(f"⏳ Waiting for job {job_id} to complete...")
        start = time.time()
        max_wait = max_wait_minutes * 60
        accepted = False

        while True:
            if time.time() - start > max_wait:
                print(f"⏰ Timeout sau {max_wait_minutes} phút")
                return None

            st = self.check_job_status(job_id)
            state = st.get("jobState", "UNKNOWN")
            print(f"   Status: {state} (elapsed: {int(time.time() - start)}s)")

            if state == "DONE":
                print("✅ Job completed.")
                return st
            if state in ["ERROR", "REJECTED", "CANCELLED"]:
                print(f"❌ Job failed: {state}")
                return None
            if state == "NEED_CONFIRMATION" and not accepted:
                if not self._accept_job(job_id):
                    return None
                accepted = True

            time.sleep(poll_seconds)
    
    # -------------------- DOWNLOAD RESULTS --------------------
    @staticmethod
    def _maybe_gunzip_bytes(data: bytes) -> bytes:
        # Thử giải nén gzip (TomTom thường nén JSON/GeoJSON, đôi khi không gắn .gz)
        try:
            return gzip.decompress(data)
        except OSError:
            return data

    @staticmethod
    def _safe_filename_from_url(url: str) -> str:
        path = unquote(urlparse(url).path)  # xử lý %2F, v.v.
        return os.path.basename(path)

    def _download_stream(self, url: str, dst_path: str):
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(dst_path, "wb") as f:
                shutil.copyfileobj(r.raw, f)
        return dst_path

    def download_results(self, job_id: str) -> Optional[dict]:
        status = self.check_job_status(job_id)
        if status.get("jobState") != "DONE":
            print(f"⚠️ Job chưa DONE: {status.get('jobState')}")
            return None

        urls = status.get("urls", [])
        if not urls:
            print("❌ Không có URL kết quả.")
            return None

        saved = {}
        for u in urls:
            fname = self._safe_filename_from_url(u)
            local = os.path.join(self.output_dir, f"{job_id}_{fname}")
            print("↓ Download:", fname)
            self._download_stream(u, local)

            lower = fname.lower()
            if lower.endswith(".json") or lower.endswith(".geojson"):
                with open(local, "rb") as f:
                    raw = f.read()
                unzipped = self._maybe_gunzip_bytes(raw)
                if unzipped != raw:
                    uz_path = local.replace(".json", "_unzipped.json").replace(".geojson", "_unzipped.geojson")
                    with open(uz_path, "wb") as fo:
                        fo.write(unzipped)
                    key = "json" if uz_path.endswith(".json") else "geojson"
                    saved[key] = uz_path
                    print("✅ Unzipped →", uz_path)
                else:
                    key = "json" if lower.endswith(".json") else "geojson"
                    saved[key] = local
            elif lower.endswith(".zip") or lower.endswith(".shp.zip") or lower.endswith(".shape"):
                saved["shape_zip"] = local
            else:
                saved[fname] = local

        status_path = os.path.join(self.output_dir, f"{job_id}_status.json")
        with open(status_path, "w", encoding="utf-8") as f:
            json.dump(status, f, indent=2, ensure_ascii=False)
        saved["status"] = status_path

        print("📁 Saved files:", json.dumps(saved, indent=2, ensure_ascii=False))
        return saved

In [29]:
import os

load_dotenv(find_dotenv())
API_KEY = os.getenv("TOMTOM_TRAFFIC_STATS_API_KEY", None)

collector = TomTomTrafficDataCollector(API_KEY)

# Ví dụ: polygon nhỏ quanh Thủ Đức
d1_geometry = {
        "type": "Polygon",
        "coordinates": [[
            [106.689091, 10.801532],
            [106.689602, 10.799872],
            [106.689997, 10.798161],
            [106.690633, 10.796125],
            [106.692143, 10.793720],
            [106.694312, 10.791469],
            [106.696253, 10.789566],
            [106.698508, 10.787826],
            [106.700826, 10.786365],
            [106.704397, 10.784622],
            [106.707994, 10.783673],
            [106.711498, 10.783168],
            [106.713733, 10.782888],
            [106.717067, 10.782710],
            [106.719835, 10.782907],
            [106.722797, 10.783626],
            [106.725503, 10.784857],
            [106.727875, 10.786677],
            [106.729745, 10.788969],
            [106.731014, 10.791355],
            [106.731689, 10.793865],
            [106.731823, 10.796180],
            [106.731344, 10.798670],
            [106.730493, 10.800875],
            [106.729225, 10.802909],
            [106.727513, 10.804745],
            [106.725374, 10.806354],
            [106.722797, 10.807715],
            [106.719835, 10.808806],
            [106.716676, 10.809534],
            [106.713133, 10.809916],
            [106.709398, 10.809830],
            [106.705994, 10.809186],
            [106.702684, 10.808086],
            [106.700112, 10.806796],
            [106.697478, 10.804974],
            [106.695282, 10.803013],
            [106.693391, 10.801299],
            [106.691470, 10.800080],
            [106.689091, 10.801532]
        ]]
    }

# Tạo job phân tích cho tháng 8/2024
# job_id = collector.create_area_analysis_job(
#         geometry=d1_geometry,
#         date_from="2024-08-01",
#         date_to="2024-08-31",
#         job_name="HCMC D1 Weekday 15min"
#     )
job_id = "7987791"
if job_id:
    status = collector.wait_for_job_completion(job_id, max_wait_minutes=60, poll_seconds=20)
    if status and status.get("jobState") == "DONE":
        files = collector.download_results(job_id)
    else:
        print("⚠️ Job chưa hoàn tất hoặc thất bại.")

💾 Output directory set to: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\tomtom_stats
⏳ Waiting for job 7987791 to complete...
   Status: DONE (elapsed: 1s)
✅ Job completed.
↓ Download: HCMC_D1_Weekday_15min.geojson
✅ Unzipped → C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\tomtom_stats\7987791_HCMC_D1_Weekday_15min_unzipped.geojson
↓ Download: HCMC_D1_Weekday_15min.json
✅ Unzipped → C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\tomtom_stats\7987791_HCMC_D1_Weekday_15min_unzipped.json
↓ Download: HCMC_D1_Weekday_15min.shapefile.zip
📁 Saved files: {
  "geojson": "C:\\AI\\Specialized_Project2_github\\Urban-Traffic-Links\\data\\raw\\tomtom_stats\\7987791_HCMC_D1_Weekday_15min_unzipped.geojson",
  "json": "C:\\AI\\Specialized_Project2_github\\Urban-Traffic-Links\\data\\raw\\tomtom_stats\\7987791_HCMC_D1_Weekday_15min_unzipped.json",
  "shape_zip": "C:\\AI\\Specialized_Project2_github\\Urban-Traffic-Links\\data\\raw\\tomtom_stats\\7987791_H

In [30]:
import os, pathlib
print("CWD =", os.getcwd())
print("OUTPUT_DIR (abs) =", pathlib.Path("./data/raw/tomtom_stats").resolve())
print("Files =", list(pathlib.Path("./data/raw/tomtom_stats").glob("*")))


CWD = C:\AI\Specialized_Project2_github\Urban-Traffic-Links\src\data_processing
OUTPUT_DIR (abs) = C:\AI\Specialized_Project2_github\Urban-Traffic-Links\src\data_processing\data\raw\tomtom_stats
Files = []
